# Notebook 3: Leveling

When one person is booked for more than 100 % of their time, the plain schedule is not realistic. Leveling fixes this by **delaying** tasks until the person is free. In this notebook we first try it on a tiny example, then on the team project from notebook 2, and finally ask whether reassigning work would do better.

## Part A: a small example

We import the library and a few helpers. The helpers only format dates, money and percentages for printing.

In [ ]:
import project_planner as pp
from datetime import date

def fmt_dt(d):
    return d.strftime("%Y-%m-%d %H:%M")

def fmt_money(x):
    return f"{x:.2f}"

def peak(ws, person):
    """Highest load of one person, in whole percent."""
    return int(max(s.percent for s in ws.loading(person).segments))

Two people, Ana and Ben, and four tasks. Ana is asked to do tasks A, B and D at the same time. Task C needs Ben, but only after B is finished.

In [ ]:
ws = pp.open_workspace(":memory:")
ws.new_project("Leveling demo", start=date(2026, 10, 5))
ana = ws.add_resource("Ana", hourly_rate="90")
ben = ws.add_resource("Ben", hourly_rate="80")

a = ws.add_task("A", duration="3d")
b = ws.add_task("B", duration="2d")
c_ = ws.add_task("C", duration="2d")
d = ws.add_task("D", duration="1d")
ws.set_assignment(a, ana, 60)
ws.set_assignment(b, ana, 60)
ws.set_assignment(c_, ben, 100)
ws.set_assignment(d, ana, 50)
ws.add_dependency(b, c_, "FS")

names = {a: "A", b: "B", c_: "C", d: "D"}

First the normal schedule, which ignores the fact that people have limited time. Look at Ana's loading: on the first day she is at 60 + 60 + 50 = 170 %.

In [ ]:
before = ws.schedule()
print(f"Before: finish {fmt_dt(before.project_finish)} | Ana peak {peak(ws, ana)}% | cost {fmt_money(before.total_cost)}")
ws.loading(ana)

`level_preview()` works out what leveling *would* do, without changing your plan. The current result is still the plain dependency schedule, as the `kind` below shows. Each delay has a cause: `resource` means the task waits for a busy person, `dependency` means it simply follows a task that was delayed.

In [ ]:
preview = ws.level_preview()
print("Current result kind:", ws.result().kind)
parts = []
for delay in sorted(preview.delays, key=lambda x: names[x.task_id]):
    days = preview.delays_days[delay.task_id].normalize()   # already in working days
    parts.append(f"{names[delay.task_id]} {days} days ({delay.cause})")
print("Delays: " + ", ".join(parts))

Now we accept the preview with `apply_leveling()` and compare. Ana never exceeds 100 %, the finish moves later, and the cost does not change because the same work is done.

In [ ]:
ws.apply_leveling()
after = ws.result()
print(f"After: finish {fmt_dt(after.project_finish)} | Ana peak {peak(ws, ana)}% | cost {fmt_money(after.total_cost)}")
print("Result kind:", after.kind)
after

**What leveling never does.** It only moves whole tasks later. It never splits a task into pieces, never changes an assignment percentage, and never reassigns work to another person. Those are planning decisions that stay with you (spec section 10).

Leveling is reversible. `reset_to_dependency_schedule()` drops it and brings the original dates back, without recalculating.

In [ ]:
restored = ws.reset_to_dependency_schedule()
print(f"Restored: finish {fmt_dt(restored.project_finish)} (kind {restored.kind})")

## Part B: the team project from notebook 2

We need this project several times, so we wrap it in a helper that builds a fresh copy each time. It has four people and eight tasks. Ana is the bottleneck.

In [ ]:
def build_team():
    ws = pp.open_workspace(":memory:")
    ws.new_project("Team project", start=date(2026, 10, 5))
    people = {n: ws.add_resource(n, hourly_rate=r)
              for n, r in [("Ana", "90"), ("Ben", "80"), ("Chloe", "70"), ("Dev", "60")]}
    spec = [
        ("Requirements", "5d", [("Ana", 100)], None),
        ("Architecture", "5d", [("Ana", 50), ("Ben", 50)], None),
        ("UX design", "5d", [("Ana", 50), ("Chloe", 100)], None),
        ("API build", "5d", [("Ben", 100)], "Architecture"),
        ("Frontend build", "5d", [("Chloe", 100), ("Ana", 50)], "UX design"),
        ("Test plan", "5d", [("Ana", 50)], None),
        ("Data migration", "5d", [("Ana", 100)], "Requirements"),
        ("Release notes", "2d", [("Dev", 25)], "API build"),
    ]
    tasks = {}
    for name, dur, assigns, pred in spec:
        tasks[name] = ws.add_task(name, duration=dur)
        for person, pct in assigns:
            ws.set_assignment(tasks[name], people[person], pct)
        if pred:
            ws.add_dependency(tasks[pred], tasks[name], "FS")
    return ws, people, tasks

def peaks(ws, people):
    return {n: peak(ws, r) for n, r in people.items()}

def peaks_text(p):
    return ", ".join(f"{n} {v}%" for n, v in p.items())

Schedule the team project as it is, ignoring overloads.

In [ ]:
ws, people, tasks = build_team()
r = ws.schedule()
print(f"Team before: finish {fmt_dt(r.project_finish)} | cost {fmt_money(r.total_cost)}")
print("Peaks before:", peaks_text(peaks(ws, people)))

Now level it. We check in code that nobody is above 100 % before we say so.

In [ ]:
ws.level_preview()
ws.apply_leveling()
r = ws.result()
p = peaks(ws, people)
assert max(p.values()) <= 100
print(f"Team leveled: finish {fmt_dt(r.project_finish)} | max peak 100% or less | cost {fmt_money(r.total_cost)}")
print("Peaks after:", peaks_text(p))

## What if we reassign instead?

Leveling can only delay. Let us try changing who does the work. Each step starts from a fresh copy of the team project.

### Step 1: move the wrong work

Move "Test plan" from Ana (50 %) to Dev (50 %), then schedule and level.

In [ ]:
def level_after(ws):
    ws.schedule()
    ws.level_preview()
    ws.apply_leveling()
    return ws.result()

ws, people, tasks = build_team()
ws.remove_assignment(tasks["Test plan"], people["Ana"])
ws.set_assignment(tasks["Test plan"], people["Dev"], 50)
r = level_after(ws)
p = peaks(ws, people)
assert max(p.values()) <= 100
print(f"Move Test plan to Dev + level: finish {fmt_dt(r.project_finish)} | cost {fmt_money(r.total_cost)}")
print("Peaks after:", peaks_text(p))

**Lesson:** the cost dropped (Dev is cheaper than Ana) but the finish did not move. Ana is still at 200 % on Requirements, Architecture and UX design, so relief only helps if it removes load from the bottleneck's critical stretch.

### Step 2: move the bottleneck's work

Now move "Data migration" from Ana (100 %) to Dev (100 %) instead.

In [ ]:
ws, people, tasks = build_team()
ws.remove_assignment(tasks["Data migration"], people["Ana"])
ws.set_assignment(tasks["Data migration"], people["Dev"], 100)
r = level_after(ws)
p = peaks(ws, people)
assert max(p.values()) <= 100
print(f"Move Data migration to Dev + level: finish {fmt_dt(r.project_finish)} | cost {fmt_money(r.total_cost)}")
print("Peaks after:", peaks_text(p))

**Lesson:** this finishes 3 working days earlier than plain leveling and costs less, since 40 hours move from 90 to 60 per hour. Leveling only delays; reassigning is a planning decision the tool leaves to you (spec section 10).